# POC demo: frozen test sample

Walk through the six-product pricing POC on **held-out people** using already-frozen artifacts. No GPU and no retraining.

Each demo person is shown as:

```text
Waves 1–3 persona  +  product question  →  LoRA prediction
                                      vs  T2 (Wave 4)  and  train-only majority
```

Run from the repository root after `poc/run_poc.sh` (or the staged commands in `poc/README.md`):

```bash
jupyter notebook notebooks/02_poc_demo_test_sample.ipynb
```

Artifacts required: `poc/artifacts/pricing/{test_inputs,test_labels,predictions,baseline,metrics}.json(l)`.
Write-up: `reports/D6_poc_training.md`.

Answers: **1 = would buy**, **2 = would not**.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

ROOT = Path.cwd()
if not (ROOT / "poc" / "artifacts" / "pricing").exists():
    if (ROOT.parent / "poc" / "artifacts" / "pricing").exists():
        ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ART = ROOT / "poc" / "artifacts" / "pricing"
for name in ("test_inputs.jsonl", "test_labels.jsonl", "predictions.jsonl", "baseline.json", "metrics.json"):
    path = ART / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run poc/run_poc.sh first.")

print(f"Repository root: {ROOT}")
print(f"Artifacts: {ART}")

Repository root: /home/user/PycharmProjects/Twin-2K-500-Assignment/Digital-Twin-Simulation
Artifacts: /home/user/PycharmProjects/Twin-2K-500-Assignment/Digital-Twin-Simulation/poc/artifacts/pricing


## 1. Load the frozen test set

`test_inputs.jsonl` has no answers. Labels are joined here only for the demo, matching `evaluate_pricing_poc.py`.

In [2]:
def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def yn(code) -> str:
    if code == 1:
        return "1 Yes"
    if code == 2:
        return "2 No"
    return str(code)


inputs = {(int(r["pid"]), r["item_key"]): r for r in read_jsonl(ART / "test_inputs.jsonl")}
labels = {(int(r["pid"]), r["item_key"]): r for r in read_jsonl(ART / "test_labels.jsonl")}
preds = {(int(r["pid"]), r["item_key"]): r for r in read_jsonl(ART / "predictions.jsonl")}
baseline_cfg = json.loads((ART / "baseline.json").read_text(encoding="utf-8"))
metrics = json.loads((ART / "metrics.json").read_text(encoding="utf-8"))
majority = {str(k): int(v) for k, v in baseline_cfg["per_item_majority"].items()}

if set(inputs) != set(labels) or set(inputs) != set(preds):
    raise RuntimeError("test_inputs / test_labels / predictions keys do not match")

rows = []
for key in sorted(inputs):
    inp, lab, pr = inputs[key], labels[key], preds[key]
    pred = pr.get("prediction")
    t1, t2 = int(lab["t1"]), int(lab["t2"])
    maj = majority[key[1]]
    rows.append(
        {
            "pid": key[0],
            "category": lab["category"],
            "product": lab["product"],
            "price": lab["price"],
            "qid": inp["qid"],
            "persona": inp["persona"],
            "question": inp["target_question"],
            "model": pred,
            "t1": t1,
            "t2": t2,
            "majority": maj,
            "model_eq_t2": pred == t2,
            "majority_eq_t2": maj == t2,
            "t1_eq_t2": t1 == t2,
        }
    )

df = pd.DataFrame(rows)
print(f"Test rows: {len(df):,}   people: {df['pid'].nunique():,}   products: {df['category'].nunique()}")

Test rows: 1,848   people: 308   products: 6


## 2. Headline scores

Same numbers as `poc/artifacts/pricing/metrics.json`.

In [3]:
ci = metrics["paired_pid_bootstrap_delta_95ci"]
headline = pd.DataFrame(
    [
        ["Global majority vs T2", f"{100 * metrics['global_majority_baseline_t2_accuracy']:.2f}%"],
        ["Item majority vs T2", f"{100 * metrics['item_majority_baseline_t2_accuracy']:.2f}%"],
        ["Qwen2.5-0.5B LoRA vs T2", f"{100 * metrics['primary_model_t2_accuracy']:.2f}%"],
        ["Δ model − item majority", f"{100 * metrics['model_over_item_baseline_delta']:+.2f} pp"],
        ["PID-bootstrap 95% CI", f"[{100 * ci[0]:+.2f}, {100 * ci[1]:+.2f}] pp"],
        ["Same preds vs T1", f"{100 * metrics['secondary_model_t1_accuracy']:.2f}%"],
        ["Human T1↔T2", f"{100 * metrics['human_t1_t2_reliability']:.2f}%"],
        ["Invalid outputs", f"{100 * metrics['invalid_output_rate']:.1f}%"],
    ],
    columns=["Metric", "Value"],
)
display(headline)

by_item = []
for item_key, stats in metrics["by_item"].items():
    by_item.append(
        {
            "category": stats["category"],
            "n": stats["n"],
            "model": f"{100 * stats['model_t2_accuracy']:.2f}%",
            "item majority": f"{100 * stats['baseline_t2_accuracy']:.2f}%",
            "human T1↔T2": f"{100 * stats['human_t1_t2_accuracy']:.2f}%",
        }
    )
display(pd.DataFrame(by_item).sort_values("model", ascending=False).reset_index(drop=True))

,Metric,Value
0,Global majority vs T2,50.16%
1,Item majority vs T2,62.07%
2,Qwen2.5-0.5B LoRA vs T2,75.97%
3,Δ model − item majority,+13.91 pp
4,PID-bootstrap 95% CI,"[+11.31, +16.56] pp"
5,Same preds vs T1,75.92%
6,Human T1↔T2,85.01%
7,Invalid outputs,0.0%


,category,n,model,item majority,human T1↔T2
0,Fresh Eggs,308,80.84%,80.84%,89.29%
1,Bottled Water,308,79.55%,70.78%,82.14%
2,Fresh Fruit,308,79.22%,54.87%,85.06%
3,Pain Remedies - Headache,308,72.73%,49.35%,82.79%
4,Soft Drinks - Carbonated,308,72.40%,67.86%,87.99%
5,Cereal - Ready to Eat,308,71.10%,48.70%,82.79%


## 3. Three test people

Change `DEMO_PIDS` to inspect anyone in the 308-person test split.

| PID | Why this person |
|---|---|
| **1** | Model matches all six T2 answers; beats majority on Tylenol (person said Yes, population No). |
| **68** | Model beats majority on four items, but T1 and T2 disagree on three — Wave 4 is not a copy of T1. |
| **515** | Miss: model is right on 1/6 T2 answers, worse than majority. |

In [4]:
DEMO_PIDS = [1, 68, 515]
missing = [pid for pid in DEMO_PIDS if pid not in set(df["pid"])]
if missing:
    raise ValueError(f"PID(s) not in the test split: {missing}")


def persona_head(text: str, n_lines: int = 8) -> str:
    lines = [ln for ln in text.splitlines() if ln.strip()]
    shown = lines[:n_lines]
    extra = len(lines) - len(shown)
    body = "\n".join(shown)
    if extra > 0:
        body += f"\n… {extra} more facts omitted"
    return body


for pid in DEMO_PIDS:
    sub = df[df["pid"] == pid].sort_values("category").reset_index(drop=True)
    n_m = int(sub["model_eq_t2"].sum())
    n_b = int(sub["majority_eq_t2"].sum())
    n_h = int(sub["t1_eq_t2"].sum())
    display(Markdown(
        f"### PID {pid}  ·  model {n_m}/6 vs T2  ·  majority {n_b}/6  ·  human T1↔T2 {n_h}/6"
    ))
    print(persona_head(sub.iloc[0]["persona"]))
    view = sub.assign(
        model=sub["model"].map(yn),
        t1=sub["t1"].map(yn),
        t2=sub["t2"].map(yn),
        majority=sub["majority"].map(yn),
        hit=sub["model_eq_t2"].map({True: "yes", False: "no"}),
        beat_maj=(
            sub["model_eq_t2"] & ~sub["majority_eq_t2"]
        ).map({True: "yes", False: ""}),
    )[["category", "product", "price", "model", "majority", "t1", "t2", "hit", "beat_maj"]]
    display(view)

### PID 1  ·  model 6/6 vs T2  ·  majority 5/6  ·  human T1↔T2 6/6

[Demographics] QID11 | Which part of the United States do you currently live in? -> 2 | Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)
[Demographics] QID12 | What is the sex that you were assigned at birth? -> 2 | Female
[Demographics] QID13 | How old are you? -> 2 | 30-49
[Demographics] QID14 | What is the highest level of schooling or degree that you have completed? -> 3 | Some college, no degree
[Demographics] QID15 | What is your race or origin? -> 2 | Black
[Demographics] QID16 | Are you a citizen of the United States? -> 1 | Yes
[Demographics] QID17 | Which of these best describes you? -> 1 | Married
[Demographics] QID18 | What is your present religion, if any? -> 12 | Nothing in particular
… 13 more facts omitted


,category,product,price,model,majority,t1,t2,hit,beat_maj
0,Bottled Water,"OZARKA Brand 100% Natural Spring Water, 16.9-ounce plastic bottles (Pack of 35)",11.98,2 No,2 No,2 No,2 No,yes,
1,Cereal - Ready to Eat,"Cinnamon Toast Crunch Breakfast Cereal, Crispy Cinnamon Cereal, Family Size,...",9.86,2 No,2 No,2 No,2 No,yes,
2,Fresh Eggs,"Eggland's Best Classic Extra Large White Eggs, 12 count",3.82,1 Yes,1 Yes,1 Yes,1 Yes,yes,
3,Fresh Fruit,"Fresh Raspberries, 12 oz Container",0.95,1 Yes,1 Yes,1 Yes,1 Yes,yes,
4,Pain Remedies - Headache,"Tylenol Extra Strength Caplets with 500 mg Acetaminophen, 100 Ct",2.19,1 Yes,2 No,1 Yes,1 Yes,yes,yes
5,Soft Drinks - Carbonated,"Coca-Cola Soda Pop, 12 fl oz, 12 Pack Cans",11.56,2 No,2 No,2 No,2 No,yes,


### PID 68  ·  model 4/6 vs T2  ·  majority 0/6  ·  human T1↔T2 3/6

[Demographics] QID11 | Which part of the United States do you currently live in? -> 3 | South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)
[Demographics] QID12 | What is the sex that you were assigned at birth? -> 1 | Male
[Demographics] QID13 | How old are you? -> 2 | 30-49
[Demographics] QID14 | What is the highest level of schooling or degree that you have completed? -> 5 | College graduate/some postgrad
[Demographics] QID15 | What is your race or origin? -> 1 | White
[Demographics] QID16 | Are you a citizen of the United States? -> 1 | Yes
[Demographics] QID17 | Which of these best describes you? -> 1 | Married
[Demographics] QID18 | What is your present religion, if any? -> 2 | Roman Catholic
… 13 more facts omitted


,category,product,price,model,majority,t1,t2,hit,beat_maj
0,Bottled Water,"OZARKA Brand 100% Natural Spring Water, 16.9-ounce plastic bottles (Pack of 35)",3.99,1 Yes,2 No,2 No,1 Yes,yes,yes
1,Cereal - Ready to Eat,"Cinnamon Toast Crunch Breakfast Cereal, Crispy Cinnamon Cereal, Family Size,...",0.00,1 Yes,2 No,1 Yes,1 Yes,yes,yes
2,Fresh Eggs,"Eggland's Best Classic Extra Large White Eggs, 12 count",5.72,1 Yes,1 Yes,1 Yes,2 No,no,
3,Fresh Fruit,"Fresh Raspberries, 12 oz Container",7.58,2 No,1 Yes,1 Yes,2 No,yes,yes
4,Pain Remedies - Headache,"Tylenol Extra Strength Caplets with 500 mg Acetaminophen, 100 Ct",8.78,1 Yes,2 No,1 Yes,1 Yes,yes,yes
5,Soft Drinks - Carbonated,"Coca-Cola Soda Pop, 12 fl oz, 12 Pack Cans",8.26,2 No,2 No,1 Yes,1 Yes,no,


### PID 515  ·  model 1/6 vs T2  ·  majority 4/6  ·  human T1↔T2 4/6

[Demographics] QID11 | Which part of the United States do you currently live in? -> 1 | Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)
[Demographics] QID12 | What is the sex that you were assigned at birth? -> 1 | Male
[Demographics] QID13 | How old are you? -> 3 | 50-64
[Demographics] QID14 | What is the highest level of schooling or degree that you have completed? -> 5 | College graduate/some postgrad
[Demographics] QID15 | What is your race or origin? -> 3 | Asian
[Demographics] QID16 | Are you a citizen of the United States? -> 1 | Yes
[Demographics] QID17 | Which of these best describes you? -> 2 | Living with a partner
[Demographics] QID18 | What is your present religion, if any? -> 9 | Atheist
… 13 more facts omitted


,category,product,price,model,majority,t1,t2,hit,beat_maj
0,Bottled Water,"OZARKA Brand 100% Natural Spring Water, 16.9-ounce plastic bottles (Pack of 35)",23.95,2 No,2 No,1 Yes,1 Yes,no,
1,Cereal - Ready to Eat,"Cinnamon Toast Crunch Breakfast Cereal, Crispy Cinnamon Cereal, Family Size,...",2.96,1 Yes,2 No,2 No,2 No,no,
2,Fresh Eggs,"Eggland's Best Classic Extra Large White Eggs, 12 count",3.18,1 Yes,1 Yes,1 Yes,1 Yes,yes,
3,Fresh Fruit,"Fresh Raspberries, 12 oz Container",7.58,2 No,1 Yes,2 No,1 Yes,no,
4,Pain Remedies - Headache,"Tylenol Extra Strength Caplets with 500 mg Acetaminophen, 100 Ct",13.16,2 No,2 No,2 No,1 Yes,no,
5,Soft Drinks - Carbonated,"Coca-Cola Soda Pop, 12 fl oz, 12 Pack Cans",1.65,1 Yes,2 No,2 No,2 No,no,


## 4. One question, fully spelled out

PID 1 / Tylenol is the case where the person-specific prediction disagrees with the train-only majority and still matches T2.

In [5]:
example = df[(df["pid"] == 1) & (df["category"] == "Pain Remedies - Headache")].iloc[0]
print(example["question"])
print()
print("Options:  1 Yes, I would purchase    2 No, I would not purchase")
print()
print(f"Train-only item majority: {yn(example['majority'])}")
print(f"LoRA prediction:          {yn(example['model'])}")
print(f"Human T1 (original):      {yn(example['t1'])}")
print(f"Human T2 (Wave 4):        {yn(example['t2'])}")

Please consider the following product category: Pain Remedies - Headache. Suppose you are in a grocery store, and you see the following product in that category: Tylenol Extra Strength Caplets with 500 mg Acetaminophen, 100 Ct. The product is priced at: $2.19. Would you or would you not purchase this product?

Options:  1 Yes, I would purchase    2 No, I would not purchase

Train-only item majority: 2 No
LoRA prediction:          1 Yes
Human T1 (original):      1 Yes
Human T2 (Wave 4):        1 Yes


This notebook does not retrain the adapter and does not claim the six-item slice is the 17-task Twin-2K-500 paper benchmark.